In [1]:
import os, json
import numpy as np, pandas as pd
from pathlib import Path

root = Path("/kaggle/input")
MAX = 40            # nombre max de fichiers affichés par dossier

def resume(p):
    ext = p.suffix.lower()
    try:
        if ext == ".npy":
            a = np.load(p, mmap_mode="r", allow_pickle=False)
            extra = ""
            if a.size and a.dtype.kind == "f":
                extra = f" | NaN={bool(np.isnan(a).any())} | min={float(a.min()):.3g} max={float(a.max()):.3g}"
            return f"npy shape={a.shape} dtype={a.dtype}{extra}"
        if ext == ".csv":
            d = pd.read_csv(p, nrows=3)
            n = sum(1 for _ in open(p, errors="ignore")) - 1
            return f"csv {n} lignes | colonnes={list(d.columns)[:12]}"
        if ext == ".json":
            j = json.load(open(p))
            if isinstance(j, dict):
                vus = {}
                for k, v in list(j.items())[:12]:
                    vus[k] = len(v) if (hasattr(v, "__len__") and not isinstance(v, str)) else v
                return f"json clés (taille ou valeur) = {vus}"
            return f"json {type(j).__name__} len={len(j)}"
        if ext in (".md", ".txt"):
            return "texte : " + open(p, errors="ignore").read(150).replace("\n", " ")
    except Exception as e:
        return f"(erreur de lecture : {e})"
    return ""

for dirpath, dirnames, filenames in sorted(os.walk(root)):
    if not filenames:
        continue
    d = Path(dirpath)
    print("\n=== ", d.relative_to(root), f"({len(filenames)} fichiers)")
    for f in sorted(filenames)[:MAX]:
        p = d / f
        print(f"  {f} | {p.stat().st_size / 1024:.0f} Ko | {resume(p)}")
    if len(filenames) > MAX:
        print(f"  ... et {len(filenames) - MAX} autres fichiers")


===  datasets/rnabenahmed/beamss (1 fichiers)
  beams.csv | 30 Ko | csv 3981 lignes | colonnes=['index', 'unit1_beam']

===  datasets/rnabenahmed/embeddings-lidar-n (39 fichiers)
  seed0.json | 1 Ko | json clés (taille ou valeur) = {'exp_id': 'LID-E2', 'seed': 0, 'n': 598, 'top1': 0.10535117238759995, 'overhead_top1': 0.015625, 'gain_top1': 0.984375, 'top3': 0.37290969491004944, 'overhead_top3': 0.046875, 'gain_top3': 0.953125, 'top5': 0.49163880944252014, 'overhead_top5': 0.078125, 'gain_top5': 0.921875}
  seed0_emb_test.npy | 1196 Ko | npy shape=(598, 512) dtype=float32 | NaN=False | min=0 max=6.55
  seed0_emb_train.npy | 5562 Ko | npy shape=(2781, 512) dtype=float32 | NaN=False | min=0 max=7.66
  seed0_emb_val.npy | 1204 Ko | npy shape=(602, 512) dtype=float32 | NaN=False | min=0 max=6.26
  seed0_embeddings.npy | 1196 Ko | npy shape=(598, 512) dtype=float32 | NaN=False | min=0 max=6.55
  seed0_idx_test.npy | 5 Ko | npy shape=(598,) dtype=int64
  seed0_idx_train.npy | 22 Ko | npy sh

# FUS-F4a — Fusion par attention : GPS + LiDAR

## Objectif
Prédire le meilleur faisceau (parmi 64) en fusionnant les **embeddings** de deux modalités déjà
entraînées et figées (GPS-E3 et LID-E2), avec un petit réseau d'attention entraîné ici.
Seul ce réseau de fusion est entraîné : les modèles GPS et LiDAR ne changent pas.

## Ce que chaque modalité apporte et ses limites
| | GPS (GPS-E3, GRU sur (x, y)) | LiDAR (LID-E2) |
|---|---|---|
| Apporte | La position exacte du véhicule, donc l'angle et la distance vus depuis la station (corrélation de l'angle avec le faisceau : 0,91) | Une perception de la scène depuis la station : position, distance et présence du véhicule, sans aide du véhicule, y compris de nuit |
| Limites | Plafond d'environ 57 % en Top-1 avec la position seule (labels ambigus entre faisceaux voisins) ; exige que le véhicule envoie sa position à la station en moins d'environ 0,1 s ; bruit et positions interpolées | Coûteux et sensible à la météo (énoncé) ; sur nos données, LID-E2 seul est faible (Top-3 d'environ 21 %) |

Les deux modalités se recouvrent en partie (position et distance), mais elles ne sont pas
soumises aux mêmes pannes : le LiDAR ne dépend pas du lien véhicule → station, le GPS ne dépend
pas de la qualité du nuage de points.

## But de la fusion
Comprendre comment chaque modalité complète l'autre : le réseau d'attention apprend, pour chaque
ligne, combien se fier au GPS et au LiDAR. C'est la première étape avant d'ajouter la caméra.

## Hypothèses (à tester, non établies)
- **H1** : la fusion GPS + LiDAR est meilleure que le GPS seul en Top-3 et en K90 (moins de
  faisceaux à tester pour 90 % de précision). Référence : GPS-E3 seul, K90 = 5.
- **H2** : quand la position GPS arrive en retard (voir `cout_retard.csv`), la fusion se dégrade
  moins que le GPS seul, parce que le LiDAR compense. À tester dans une étape ultérieure.
- **H3** : les poids d'attention donnent plus de place au LiDAR quand le GPS est peu fiable.
- **Résultat nul possible** : si le LiDAR n'apporte rien, le réseau apprendra à l'ignorer et le
  résultat sera proche de celui du GPS seul. Ce serait aussi un résultat à rapporter.

## Bénéfice attendu
Un K90 plus petit que celui du GPS seul, c'est-à-dire moins de faisceaux à tester pour la même
précision, et une moindre dépendance à un GPS frais.

## Protocole
- Entrées : embeddings GPS (128 valeurs) et LiDAR (512 valeurs), standardisés avec les
  statistiques du train.
- Réseau : projection de chaque modalité vers une taille commune, couches d'attention entre les
  modalités, couche finale vers 64 faisceaux.
- Configurations : 6 au maximum, choisies sur la validation (règle : Top-3 de validation, puis perte
  de puissance). Puis le meilleur réglage est relancé sur les seeds 0, 1 et 2.
- Le test n'est évalué qu'une seule fois par seed.
- Décision d'équipe en suspens : entraîner la fusion sur le train (choix actuel) ou sur la
  validation, car les embeddings du train sont issus de modèles déjà entraînés sur ces lignes.

## Métriques
Top-1, Top-3, Top-5, perte de puissance (dB), overhead K/64, K90, K95, économie, nombre de
paramètres, temps d'entraînement. Références : GPS-E3 seul, LID-E2 seul.

In [5]:
import json, time, copy, random
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path

base = Path("/kaggle/input/datasets/rnabenahmed")
gcr = base / "gps-complet"
LID_DIR = base / "embeddings-lidar-n"                                          # CHANGÉ : nouveau dossier LiDAR
GPS_EMB_DIR = base / "gps-embeddings" / "project" / "results" / "GPS-E3"       # AJOUTÉ : embeddings GPS
GPS_RES_DIR = gcr / "project" / "results" / "GPS-E3"                           # AJOUTÉ : logits GPS
OUT = Path("/kaggle/working/results/FUS-F4a"); OUT.mkdir(parents=True, exist_ok=True)
dev = "cuda" if torch.cuda.is_available() else "cpu"
N_BEAMS = 64

# ---- splits, labels (beam0 = 0..63) et puissances
splits = json.load(open(base / "splits" / "splits.json"))
master = pd.read_csv(gcr / "ds6g" / "master.csv")
power = np.load(gcr / "ds6g" / "cache" / "power.npy")             # (3981, 64), avec des NaN
pos = {int(i): k for k, i in enumerate(master["idx"])}
y_by_idx = master.set_index("idx")["beam0"]

Y, P = {}, {}
for s in ["train", "val", "test"]:
    ids = np.array(splits[s])
    rows = [pos[int(i)] for i in ids]
    Y[s] = torch.tensor(y_by_idx.loc[ids].to_numpy(), dtype=torch.long)
    P[s] = torch.tensor(np.nan_to_num(power[rows], nan=0.0), dtype=torch.float32)   # NaN -> 0 : à aligner sur dataset.py
    print(s, Y[s].shape, P[s].shape)

# ---- métriques : copie de common/metrics.py (ne pas la modifier) + K90
KS = (1, 3, 5)

def metrics_from_logits(L, Y, P, ks=KS):
    top, pm = L.topk(max(ks), dim=1).indices, P.max(1).values
    valid = pm > 0
    res = {"n": int(len(Y))}
    for k in ks:
        res[f"top{k}"] = (top[:, :k] == Y[:, None]).any(1).float().mean().item()
        pk = P.gather(1, top[:, :k]).max(1).values.clamp(min=1e-3)
        res[f"pwr_loss_db_top{k}"] = (10 * torch.log10(pm[valid] / pk[valid])).mean().item()
        res[f"overhead_top{k}"] = k / N_BEAMS
    return res

def k_for_target(L, Y, target=0.90):
    rangs = (L > L.gather(1, Y[:, None])).sum(1) + 1
    for k in range(1, N_BEAMS + 1):
        if (rangs <= k).float().mean().item() >= target:
            return k

def full_metrics(L, Y, P):
    m = metrics_from_logits(L, Y, P)
    m["K90"] = k_for_target(L, Y, 0.90)
    m["K95"] = k_for_target(L, Y, 0.95)
    m["economie_K90"] = 1 - m["K90"] / N_BEAMS
    return m

def count_params(model):
    return sum(p.numel() for p in model.parameters())

# ---- CONTRÔLE : on doit retrouver GPS-E3, seed 0
Lg = torch.tensor(np.load(GPS_RES_DIR / "seed0_test_logits.npy"))
print(full_metrics(Lg, Y["test"], P["test"]))
# Attendu : top1=0.5435, top3=0.8495, top5=0.9281, pwr_loss_db_top1≈0.1156, pwr_loss_db_top5≈0.0445, K90=5

train torch.Size([2781]) torch.Size([2781, 64])
val torch.Size([602]) torch.Size([602, 64])
test torch.Size([598]) torch.Size([598, 64])
{'n': 598, 'top1': 0.54347825050354, 'pwr_loss_db_top1': 0.11562828719615936, 'overhead_top1': 0.015625, 'top3': 0.8494983315467834, 'pwr_loss_db_top3': 0.062350038439035416, 'overhead_top3': 0.046875, 'top5': 0.928093671798706, 'pwr_loss_db_top5': 0.044501893222332, 'overhead_top5': 0.078125, 'K90': 5, 'K95': 6, 'economie_K90': 0.921875}


In [6]:
def load_gps(seed):        # 128 dimensions ; idx communs aux seeds
    return {s: (np.load(GPS_EMB_DIR / f"{s}_idx.npy"), np.load(GPS_EMB_DIR / f"seed{seed}_{s}_emb.npy"))
            for s in ["train", "val", "test"]}                                  # CHANGÉ : GPS_EMB_DIR

def load_lidar(seed):      # 512 dimensions
    return {s: (np.load(LID_DIR / f"seed{seed}_idx_{s}.npy"), np.load(LID_DIR / f"seed{seed}_emb_{s}.npy"))
            for s in ["train", "val", "test"]}                                  # CHANGÉ : LID_DIR

def align(mods, split):
    ids = np.array(splits[split])
    out = []
    for m in mods:
        idx, emb = m[split]
        p = {int(i): k for k, i in enumerate(idx)}
        out.append(emb[[p[int(i)] for i in ids]])
    return out

def get_data(seed):
    gps, lid = load_gps(seed), load_lidar(seed)
    D = {s: align([gps, lid], s) for s in ["train", "val", "test"]}
    stats = [(x.mean(0), x.std(0) + 1e-6) for x in D["train"]]            # statistiques du train seulement
    return {s: [torch.tensor((x - m) / sd, dtype=torch.float32) for x, (m, sd) in zip(D[s], stats)]
            for s in D}

class AttnFusion(nn.Module):
    """Un jeton par modalité ; l'attention fait regarder chaque modalité aux autres."""
    def __init__(self, dims, d=64, heads=4, layers=1, dropout=0.2, n_classes=N_BEAMS):
        super().__init__()
        self.proj = nn.ModuleList([nn.Sequential(nn.Linear(di, d), nn.LayerNorm(d)) for di in dims])
        self.mod_emb = nn.Parameter(torch.randn(len(dims), d) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d, nhead=heads, dim_feedforward=2 * d,
                                           dropout=dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d * len(dims)), nn.Dropout(dropout),
                                  nn.Linear(d * len(dims), n_classes))
    def forward(self, xs):
        toks = torch.stack([p(x) for p, x in zip(self.proj, xs)], dim=1) + self.mod_emb   # (B, M, d)
        return self.head(self.enc(toks).flatten(1))

def soft_targets(Pb, Yb, tau):
    q = Pb / Pb.max(1, keepdim=True).values.clamp(min=1e-9)      # puissances normalisées par ligne (hypothèse à aligner sur E3-s)
    t = F.softmax(q / tau, dim=1)
    ok = Pb.max(1).values > 0
    return torch.where(ok[:, None], t, F.one_hot(Yb, N_BEAMS).float())

def predict(model, xs, bs=512):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(xs[0]), bs):
            out.append(model([x[i:i + bs].to(dev) for x in xs]).cpu())
    return torch.cat(out)

def train_one(cfg, data, seed, max_epochs=60, patience=8, bs=128, lr=1e-3):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    dims = [x.shape[1] for x in data["train"]]
    model = AttnFusion(dims, d=cfg["d"], heads=cfg["heads"], layers=cfg["layers"],
                       dropout=cfg["dropout"]).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    Xtr, Ytr, Ptr = data["train"], Y["train"], P["train"]
    n, best, best_state, bad, t0 = len(Ytr), None, None, 0, time.time()
    for ep in range(max_epochs):
        model.train(); perm = torch.randperm(n)
        for i in range(0, n, bs):
            b = perm[i:i + bs]
            logits = model([x[b].to(dev) for x in Xtr])
            if cfg["tau"] is None:
                loss = F.cross_entropy(logits, Ytr[b].to(dev))
            else:
                tgt = soft_targets(Ptr[b], Ytr[b], cfg["tau"]).to(dev)
                loss = -(tgt * F.log_softmax(logits, 1)).sum(1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        mv = metrics_from_logits(predict(model, data["val"]), Y["val"], P["val"])
        score = (mv["top3"], -mv["pwr_loss_db_top1"])            # règle : Top-3 de validation, puis perte de puissance
        if best is None or score > best:
            best, best_state, bad = score, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, time.time() - t0

In [7]:
# --- le LiDAR (nouveau dossier) doit retrouver ses propres chiffres
Ll = torch.tensor(np.load(LID_DIR / "seed0_test_logits.npy"))
print(full_metrics(Ll, Y["test"], P["test"]))
# Attendu (seed0.json) : top1≈0.1054, top3≈0.3729, top5≈0.4916

# --- l'ordre des échantillons LiDAR est-il celui de splits ?
for s in ["train", "val", "test"]:
    print(s, "idx LiDAR == splits :", np.array_equal(np.load(LID_DIR / f"seed0_idx_{s}.npy"), np.array(splits[s])))
    print(s, "idx GPS   == splits :", np.array_equal(np.load(GPS_EMB_DIR / f"{s}_idx.npy"), np.array(splits[s])))

# --- formes des entrées de la fusion
d0 = get_data(0)
print([x.shape for x in d0["train"]])   # attendu : [(2781, 128), (2781, 512)]
print([x.shape for x in d0["test"]])    # attendu : [(598, 128), (598, 512)]

{'n': 598, 'top1': 0.10535117238759995, 'pwr_loss_db_top1': 1.1947788000106812, 'overhead_top1': 0.015625, 'top3': 0.37290969491004944, 'pwr_loss_db_top3': 0.6241913437843323, 'overhead_top3': 0.046875, 'top5': 0.49163880944252014, 'pwr_loss_db_top5': 0.4074643552303314, 'overhead_top5': 0.078125, 'K90': 32, 'K95': 44, 'economie_K90': 0.5}
train idx LiDAR == splits : True
train idx GPS   == splits : True
val idx LiDAR == splits : True
val idx GPS   == splits : True
test idx LiDAR == splits : True
test idx GPS   == splits : True
[torch.Size([2781, 128]), torch.Size([2781, 512])]
[torch.Size([598, 128]), torch.Size([598, 512])]


In [8]:
configs = [
    dict(d=64,  heads=4, layers=1, dropout=0.2, tau=None),
    dict(d=64,  heads=4, layers=2, dropout=0.2, tau=None),
    dict(d=128, heads=4, layers=1, dropout=0.3, tau=None),
    dict(d=64,  heads=4, layers=1, dropout=0.2, tau=0.05),
    dict(d=128, heads=4, layers=2, dropout=0.3, tau=0.05),
    dict(d=64,  heads=4, layers=1, dropout=0.5, tau=0.05),
]
data0 = get_data(0)
rows = []
for k, cfg in enumerate(configs):
    model, tt = train_one(cfg, data0, seed=0)
    mv = full_metrics(predict(model, data0["val"]), Y["val"], P["val"])
    rows.append({"cfg": k, **cfg, "params": count_params(model), "train_time_s": round(tt, 1),
                 "val_top1": mv["top1"], "val_top3": mv["top3"], "val_top5": mv["top5"],
                 "val_pwr_loss_db_top1": mv["pwr_loss_db_top1"], "val_K90": mv["K90"]})
    print(rows[-1])

val_table = pd.DataFrame(rows).sort_values(["val_top3", "val_pwr_loss_db_top1"], ascending=[False, True])
val_table.to_csv(OUT / "val_configs.csv", index=False)
print(val_table.round(4).to_string())
best_cfg = configs[int(val_table.iloc[0]["cfg"])]
print("Meilleure configuration (validation) :", best_cfg)

{'cfg': 0, 'd': 64, 'heads': 4, 'layers': 1, 'dropout': 0.2, 'tau': None, 'params': 83456, 'train_time_s': 4.8, 'val_top1': 0.2425249218940735, 'val_top3': 0.5083056688308716, 'val_top5': 0.6495016813278198, 'val_pwr_loss_db_top1': 0.40046247839927673, 'val_K90': 23}
{'cfg': 1, 'd': 64, 'heads': 4, 'layers': 2, 'dropout': 0.2, 'tau': None, 'params': 116928, 'train_time_s': 3.4, 'val_top1': 0.2508305609226227, 'val_top3': 0.5066445469856262, 'val_top5': 0.6262458562850952, 'val_pwr_loss_db_top1': 0.4637690484523773, 'val_K90': 20}
{'cfg': 2, 'd': 128, 'heads': 4, 'layers': 1, 'dropout': 0.3, 'tau': None, 'params': 232384, 'train_time_s': 5.1, 'val_top1': 0.2425249218940735, 'val_top3': 0.5282391905784607, 'val_top5': 0.6661129593849182, 'val_pwr_loss_db_top1': 0.4039791226387024, 'val_K90': 21}
{'cfg': 3, 'd': 64, 'heads': 4, 'layers': 1, 'dropout': 0.2, 'tau': 0.05, 'params': 83456, 'train_time_s': 6.9, 'val_top1': 0.2807309031486511, 'val_top3': 0.6644518375396729, 'val_top5': 0.79734

In [9]:
# 1) Références sur la VALIDATION (même ensemble que la fusion)
g = pd.read_csv(GPS_RES_DIR / "final_seeds.csv")
print(g[["seed", "val_top1", "val_top3", "val_top5", "val_pwr_loss_db_top1"]].round(4))
Llv = torch.tensor(np.load(LID_DIR / "seed0_val_logits.npy"))
mv_l = full_metrics(Llv, Y["val"], P["val"])
print("LiDAR seul (val):", {k: round(mv_l[k], 4) for k in ("top1", "top3", "top5", "K90")})

# 2) Ablation sur la validation : on masque une modalité (0 = moyenne du train)
m, _ = train_one(best_cfg, data0, seed=0)
def val_top3(xs):
    return round(full_metrics(predict(m, xs), Y["val"], P["val"])["top3"], 4)
v = data0["val"]
print("GPS + LiDAR   :", val_top3(v))
print("LiDAR masqué  :", val_top3([v[0], torch.zeros_like(v[1])]))
print("GPS masqué    :", val_top3([torch.zeros_like(v[0]), v[1]]))

   seed  val_top1  val_top3  val_top5  val_pwr_loss_db_top1
0     0    0.3821    0.7425    0.8472                0.1629
1     1    0.3837    0.7259    0.8455                0.1687
2     2    0.3771    0.7276    0.8372                0.1660
LiDAR seul (val): {'top1': 0.0947, 'top3': 0.2093, 'top5': 0.2874, 'K90': 45}
GPS + LiDAR   : 0.6744
LiDAR masqué  : 0.6379
GPS masqué    : 0.1429


## FUS-F4a — Fusion par attention croisée GPS + LiDAR : configuration retenue

### Réseau de fusion (`AttnFusion`)
- **Entrées** : un embedding par modalité, issu des meilleurs modèles unimodaux (GPS-E3 : 128 dimensions ; LiDAR LID-E2 : 512 dimensions). Les embeddings sont normalisés avec les statistiques du **train uniquement**.
- **Un jeton par modalité** : projection `Linear(dim → d)` puis `LayerNorm`, plus un embedding de modalité appris.
- **Encodeur Transformer** sur les jetons (4 têtes, couche feed-forward de taille 2d, dropout), pour que chaque modalité puisse « regarder » l'autre.
- **Sortie** : concaténation des jetons, `LayerNorm`, `Dropout`, `Linear → 64` beams.

### Configuration retenue (choisie sur la validation, seed 0)

| Paramètre | Valeur |
|---|---|
| d (taille des jetons) | 128 |
| Couches d'attention | 2 |
| Têtes d'attention | 4 |
| Dropout | 0,3 |
| Labels | **doux**, tau = 0,05 |
| Paramètres | 364 864 |

### Entraînement
AdamW (lr = 1e-3, weight decay = 1e-2), batch de 128, 60 époques maximum, arrêt anticipé avec patience de 8. Le meilleur modèle est celui qui maximise le **Top-3 de validation**, puis minimise la perte de puissance.

**Labels doux** : cible = softmax(P / max(P) / tau), avec P les 64 puissances mesurées. Si la ligne n'a aucune puissance valide (NaN remplacés par 0), on revient à un label one-hot.

### Recherche d'hyperparamètres (6 configurations, validation, seed 0)

| cfg | d | couches | dropout | tau | Params | Top-1 | Top-3 | Top-5 | Perte puissance Top-1 (dB) | K90 |
|---|---|---|---|---|---|---|---|---|---|---|
| **4** | **128** | **2** | **0,3** | **0,05** | **364 864** | **0,311** | **0,674** | **0,802** | **0,235** | **16** |
| 3 | 64 | 1 | 0,2 | 0,05 | 83 456 | 0,281 | 0,664 | 0,797 | 0,229 | 15 |
| 5 | 64 | 1 | 0,5 | 0,05 | 83 456 | 0,296 | 0,664 | 0,794 | 0,284 | 15 |
| 2 | 128 | 1 | 0,3 | — | 232 384 | 0,243 | 0,528 | 0,666 | 0,404 | 21 |
| 0 | 64 | 1 | 0,2 | — | 83 456 | 0,243 | 0,508 | 0,650 | 0,401 | 23 |
| 1 | 64 | 2 | 0,2 | — | 116 928 | 0,251 | 0,507 | 0,626 | 0,464 | 20 |

**Règle de choix fixée à l'avance** : Top-3 de validation, puis perte de puissance. La configuration 4 est donc retenue.

### Ce que montre la recherche
- **Le facteur qui compte est le type de label** : avec les labels doux, le Top-3 de validation passe de 0,51-0,53 à 0,66-0,67 et la perte de puissance est presque divisée par 2.
- Entre les 3 configurations à labels doux, l'écart de Top-3 (0,674 contre 0,664) correspond à environ 6 échantillons sur 602 : il n'est **pas significatif**. La configuration 3 donne un résultat équivalent avec 4 fois moins de paramètres.
- Les hyperparamètres ont été choisis sur la seed 0 seulement ; le résultat final est évalué avec les seeds 0, 1 et 2.

### Diagnostic (validation, seed 0)

| Entrées | Top-3 val |
|---|---|
| GPS + LiDAR | 0,674 |
| LiDAR neutralisé (moyenne du train) | 0,638 |
| GPS neutralisé (moyenne du train) | *à reporter depuis la sortie* |
| GPS-E3 seul (3 seeds) | 0,726 à 0,743 |
| LiDAR seul | 0,209 |

La fusion s'appuie surtout sur le GPS et reste en dessous du GPS seul sur la validation.

In [10]:
res = []
for seed in [0, 1, 2]:
    data = get_data(seed)
    model, tt = train_one(best_cfg, data, seed=seed)
    Lv, Lt = predict(model, data["val"]), predict(model, data["test"])
    mv = full_metrics(Lv, Y["val"], P["val"])
    mt = full_metrics(Lt, Y["test"], P["test"])               # test évalué une seule fois
    rec = {"exp_id": "FUS-F4a", "seed": seed,
           **{f"val_{k}": v for k, v in mv.items()},
           **{f"test_{k}": v for k, v in mt.items()},
           "params": count_params(model), "train_time_s": round(tt, 1), "config": best_cfg}
    json.dump(rec, open(OUT / f"seed{seed}.json", "w"), indent=1)
    np.save(OUT / f"seed{seed}_val_logits.npy", Lv.numpy())
    np.save(OUT / f"seed{seed}_test_logits.npy", Lt.numpy())
    torch.save(model.state_dict(), OUT / f"seed{seed}_best.pt")
    res.append(rec)
    print(seed, "| val top3 =", round(mv["top3"], 4), "| test top1/top3/top5 =",
          round(mt["top1"], 4), round(mt["top3"], 4), round(mt["top5"], 4), "| K90 test =", mt["K90"])

pd.DataFrame(res).drop(columns="config").to_csv(OUT / "final_seeds.csv", index=False)

# ---- références : GPS seul et LiDAR seul, mêmes lignes, mêmes métriques
def gps_logits(s, split):
    return np.load(GPS_RES_DIR / f"seed{s}_{split}_logits.npy")

def lidar_logits(s, split):                                   # réordonné selon splits[split]
    idx = np.load(LID_DIR / f"seed{s}_idx_{split}.npy")
    L = np.load(LID_DIR / f"seed{s}_{split}_logits.npy")
    p = {int(i): k for k, i in enumerate(idx)}
    return L[[p[int(i)] for i in splits[split]]]

def ref(loader, split):
    return pd.DataFrame([full_metrics(torch.tensor(loader(s, split)), Y[split], P[split]) for s in [0, 1, 2]])

cols = ["top1", "top3", "top5", "pwr_loss_db_top1", "pwr_loss_db_top3", "pwr_loss_db_top5", "K90", "K95", "economie_K90"]
for split in ["val", "test"]:
    fus = pd.DataFrame([{c: r[f"{split}_{c}"] for c in cols} for r in res])
    refs = {"GPS-E3 seul": ref(gps_logits, split)[cols],
            "LID-E2 seul": ref(lidar_logits, split)[cols],
            "FUS-F4a (GPS+LiDAR)": fus}
    tab = pd.DataFrame({k: {c: f"{v[c].mean():.3f} ± {v[c].std():.3f}" for c in cols}
                        for k, v in refs.items()}).T
    print(f"\n=== {split.upper()} (moyenne ± écart-type sur 3 seeds) ===")
    print(tab.to_string())
    tab.to_csv(OUT / f"tableau_comparaison_{split}.csv")

0 | val top3 = 0.6744 | test top1/top3/top5 = 0.4448 0.7492 0.8863 | K90 test = 6
1 | val top3 = 0.6578 | test top1/top3/top5 = 0.3161 0.7609 0.8829 | K90 test = 6
2 | val top3 = 0.6744 | test top1/top3/top5 = 0.311 0.7977 0.8896 | K90 test = 6

=== VAL (moyenne ± écart-type sur 3 seeds) ===
                              top1           top3           top5 pwr_loss_db_top1 pwr_loss_db_top3 pwr_loss_db_top5             K90             K95   economie_K90
GPS-E3 seul          0.381 ± 0.003  0.732 ± 0.009  0.843 ± 0.005    0.166 ± 0.003    0.087 ± 0.000    0.069 ± 0.000  11.000 ± 0.000  21.667 ± 0.577  0.828 ± 0.000
LID-E2 seul          0.102 ± 0.007  0.216 ± 0.008  0.298 ± 0.011    1.535 ± 0.050    0.866 ± 0.021    0.643 ± 0.025  45.333 ± 2.517  56.000 ± 2.000  0.292 ± 0.039
FUS-F4a (GPS+LiDAR)  0.315 ± 0.009  0.669 ± 0.010  0.802 ± 0.003    0.236 ± 0.007    0.101 ± 0.003    0.067 ± 0.001  15.000 ± 1.000  32.667 ± 4.619  0.766 ± 0.016

=== TEST (moyenne ± écart-type sur 3 seeds) ===
      

# FUS-F4a — Fusion par attention croisée GPS + LiDAR

## Hypothèse
Le GPS donne la position (x, y) du véhicule, et le beam dépend de son angle par rapport à la station (corrélation de Spearman de 0,91 avec le beam). Le LiDAR perçoit la scène autour de la station. **Hypothèse testée** : fusionner les deux par attention croisée améliore la prédiction du beam par rapport au GPS seul, en apportant une information que la position seule ne contient pas.

## Ce qui a été fait
- **Entrées** : les embeddings des meilleurs modèles unimodaux, normalisés avec les statistiques du train uniquement.
  - GPS-E3 : GRU sur la position actuelle (x, y), sans historique (K = 0), labels doux, 128 dimensions.
  - LID-E2 : embeddings LiDAR, 512 dimensions (nouvelle version `embeddings-lidar-n`).
- **Modèle** : un jeton par modalité, encodeur Transformer, sortie sur 64 beams.
- **Configuration** : choisie sur la validation (seed 0) parmi 6 configurations ; retenue : d = 128, 2 couches, 4 têtes, dropout 0,3, labels doux (tau = 0,05), 364 864 paramètres.
- **Évaluation finale** : seeds 0, 1 et 2 ; test évalué une seule fois. Mêmes lignes de test et mêmes métriques que pour GPS-E3 et LID-E2.
- **Effet des labels doux** : le Top-3 de validation passe de 0,51-0,53 (labels durs) à 0,66-0,67 (labels doux).

## Résultats (test, moyenne ± écart-type sur 3 seeds)

| | Top-1 | Top-3 | Top-5 | Perte puiss. Top-1 (dB) | K90 | K95 | Économie K90 |
|---|---|---|---|---|---|---|---|
| GPS-E3 seul | 0,556 ± 0,013 | 0,852 ± 0,003 | 0,928 ± 0,004 | 0,112 ± 0,004 | 5 | 6,7 ± 0,6 | 92,2 % |
| LID-E2 seul | 0,089 ± 0,025 | 0,258 ± 0,100 | 0,350 ± 0,123 | 1,802 ± 0,528 | 40 ± 12 | 47 ± 10 | 37,5 % |
| **FUS-F4a** | 0,357 ± 0,076 | 0,769 ± 0,025 | 0,886 ± 0,003 | 0,321 ± 0,051 | 6 | 11,3 ± 0,6 | 90,6 % |

Validation (3 seeds) : le même classement apparaît (Top-3 de 0,732 pour le GPS seul, 0,669 pour la fusion, 0,216 pour le LiDAR seul).

## Conclusion : l'hypothèse n'est pas confirmée
La fusion est **inférieure au GPS seul** sur toutes les métriques, avec un écart de -20 points en Top-1, -8 en Top-3 et -4 en Top-5. Elle reste très efficace face au balayage exhaustif : 6 beams sur 64 couvrent 90 % des cas, soit 90,6 % d'économie.

## Analyse (hypothèses à vérifier, rien n'est encore démontré)

**Observé (validation, seed 0, un seul modèle).**

| Entrées données au modèle | Top-3 val |
|---|---|
| GPS + LiDAR | 0,674 |
| LiDAR neutralisé (moyenne du train) | 0,638 |
| GPS neutralisé (moyenne du train) | ≈ 0,14 (valeur à reporter depuis la sortie) |

La fusion s'appuie presque entièrement sur le GPS. Même sans le LiDAR, elle reste sous le GPS seul (0,638 contre 0,732).

**Explications possibles :**
1. **Embeddings du train trop favorables.** Les encodeurs unimodaux ont été entraînés sur ces mêmes échantillons : la fusion apprend sur des entrées plus fiables que celles qu'elle voit en validation et en test. *Vérification : comparer le Top-1 du LiDAR sur le train et sur le test. Résultat : à compléter.*
2. **Trop de paramètres pour peu de données** : 364 864 paramètres pour 2781 échantillons très corrélés (13 séquences).
3. **Confusion entre beams voisins** : le Top-1 varie beaucoup d'une seed à l'autre (0,445 ; 0,316 ; 0,311) et l'écart se réduit quand K augmente. *Vérification : distribution de l'erreur en nombre de beams, GPS contre fusion. Résultat : à compléter.*

## Limites
- Le test ne contient que **2 séquences** (598 échantillons très corrélés) : les écarts-types sur 3 seeds ne mesurent que l'aléa d'entraînement, pas la variabilité entre séquences.
- LID-E2 est un modèle LiDAR faible : ce résultat ne dit rien du potentiel du LiDAR en général.
- La configuration a été choisie sur une seule seed.


# FUS-F4c — Fusion GPS + caméra

## Ce que chaque modalité apporte

| | GPS (GPS-E3) | Caméra (CAM-best) |
|---|---|---|
| **Information** | Position (x, y) du véhicule, dont dépend l'angle, qui détermine le beam | Image de la scène : on voit où se trouve le véhicule |
| **Atouts** | Précis, indépendant de l'éclairage, très prédictif (Top-3 de test de 0,852) | Perçoit l'environnement et ne dépend pas d'un récepteur GPS sur le véhicule |
| **Limites** | Erreurs de mesure : 63 lignes sans fix valide et 302 positions interpolées | Toutes les images sont de nuit ; sensible à la luminosité et aux occlusions |
| **Embeddings utilisés** | 128 dimensions | *à compléter* dimensions |
| **Résultat seul (test)** | Top-1 0,556 / Top-3 0,852 / Top-5 0,928 | *à compléter* |

## Objectif de l'expérience
Savoir si la fusion exploite une **complémentarité** entre les deux modalités : la caméra apporte-t-elle une information que le GPS ne contient pas, par exemple quand la position GPS est erronée ou quand le meilleur beam ne suit pas la géométrie ?

**Méthode.** Même protocole que FUS-F4a : mêmes splits (13/3/2 séquences), fusion par attention croisée, configuration choisie sur la validation (6 configurations maximum), seeds 0, 1 et 2, test évalué une seule fois. Comparaison sur les mêmes lignes de test :
- GPS seul, caméra seule, fusion GPS + caméra ;
- ablation : une modalité neutralisée à la fois ;
- analyse des erreurs sur les échantillons où le GPS se trompe de plus de 5 beams.

In [11]:
from pathlib import Path
import numpy as np, json

base = Path("/kaggle/input/datasets/rnabenahmed")
CAM_DIR = base / "camera-best"          # à ajuster si les fichiers sont dans un sous-dossier

for p in sorted(CAM_DIR.rglob("*")):
    if not p.is_file():
        continue
    if p.suffix == ".npy":
        a = np.load(p, mmap_mode="r")
        print(f"{str(p.relative_to(CAM_DIR)):45s} npy shape={a.shape} dtype={a.dtype}")
    elif p.suffix == ".json":
        print(f"{str(p.relative_to(CAM_DIR)):45s} json {str(json.load(open(p)))[:300]}")
    else:
        print(f"{str(p.relative_to(CAM_DIR)):45s} {p.stat().st_size // 1024} Ko")

CAM-BEST/BEST.json                            json {'best_experiment': 'CAM-E3', 'selected_by': 'mean val score (mean of top1/3/5) over seeds', 'files': 'seed{s}_{val,test,train}_logits.npy: rows in ascending master.csv row order within each split; seed{s}_embeddings.npy: (N, D) float16 for ALL samples in master.csv row order; seed{s}_{train,val,tes
CAM-BEST/seed0.json                           json {'exp_id': 'CAM-E3', 'seed': 0, 'n': 598, 'top1': 0.44147157669067383, 'pwr_loss_db_top1': 0.4401412904262543, 'overhead_top1': 0.015625, 'top3': 0.7408027052879333, 'pwr_loss_db_top3': 0.18529105186462402, 'overhead_top3': 0.046875, 'top5': 0.8177257776260376, 'pwr_loss_db_top5': 0.1256769299507141
CAM-BEST/seed0_embeddings.npy                 npy shape=(3981, 1280) dtype=float16
CAM-BEST/seed0_history.json                   json [{'epoch': 1, 'train_loss': 3.6617716523103936, 'val_loss': 4.400017738342285, 'val_top1': 0.11627907305955887, 'val_top3': 0.22425249218940735, 'val_top5': 0.308

configuration, labels, métriques, contrôle GPS

In [12]:
import json, time, copy, random
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path

base = Path("/kaggle/input/datasets/rnabenahmed")
gcr = base / "gps-complet"
CAM_DIR = next(p.parent for p in (base / "camera-best").rglob("BEST.json"))   # trouve le bon sous-dossier
GPS_EMB_DIR = base / "gps-embeddings" / "project" / "results" / "GPS-E3"
GPS_RES_DIR = gcr / "project" / "results" / "GPS-E3"
OUT = Path("/kaggle/working/results/FUS-F4c"); OUT.mkdir(parents=True, exist_ok=True)
dev = "cuda" if torch.cuda.is_available() else "cpu"
N_BEAMS = 64
print("Dossier caméra :", CAM_DIR)

# ---- splits, labels (beam0 = 0..63) et puissances
splits = json.load(open(base / "splits" / "splits.json"))
master = pd.read_csv(gcr / "ds6g" / "master.csv")
power = np.load(gcr / "ds6g" / "cache" / "power.npy")
pos = {int(i): k for k, i in enumerate(master["idx"])}
y_by_idx = master.set_index("idx")["beam0"]

Y, P = {}, {}
for s in ["train", "val", "test"]:
    ids = np.array(splits[s])
    rows = [pos[int(i)] for i in ids]
    Y[s] = torch.tensor(y_by_idx.loc[ids].to_numpy(), dtype=torch.long)
    P[s] = torch.tensor(np.nan_to_num(power[rows], nan=0.0), dtype=torch.float32)
    print(s, Y[s].shape, P[s].shape)

# ---- métriques : copie de common/metrics.py + K90
KS = (1, 3, 5)

def metrics_from_logits(L, Y, P, ks=KS):
    top, pm = L.topk(max(ks), dim=1).indices, P.max(1).values
    valid = pm > 0
    res = {"n": int(len(Y))}
    for k in ks:
        res[f"top{k}"] = (top[:, :k] == Y[:, None]).any(1).float().mean().item()
        pk = P.gather(1, top[:, :k]).max(1).values.clamp(min=1e-3)
        res[f"pwr_loss_db_top{k}"] = (10 * torch.log10(pm[valid] / pk[valid])).mean().item()
        res[f"overhead_top{k}"] = k / N_BEAMS
    return res

def k_for_target(L, Y, target=0.90):
    rangs = (L > L.gather(1, Y[:, None])).sum(1) + 1
    for k in range(1, N_BEAMS + 1):
        if (rangs <= k).float().mean().item() >= target:
            return k

def full_metrics(L, Y, P):
    m = metrics_from_logits(L, Y, P)
    m["K90"] = k_for_target(L, Y, 0.90)
    m["K95"] = k_for_target(L, Y, 0.95)
    m["economie_K90"] = 1 - m["K90"] / N_BEAMS
    return m

def count_params(model):
    return sum(p.numel() for p in model.parameters())

# ---- CONTRÔLE : on doit retrouver GPS-E3, seed 0
Lg = torch.tensor(np.load(GPS_RES_DIR / "seed0_test_logits.npy"))
print(full_metrics(Lg, Y["test"], P["test"]))
# Attendu : top1=0.5435, top3=0.8495, top5=0.9281, K90=5

Dossier caméra : /kaggle/input/datasets/rnabenahmed/camera-best/CAM-BEST
train torch.Size([2781]) torch.Size([2781, 64])
val torch.Size([602]) torch.Size([602, 64])
test torch.Size([598]) torch.Size([598, 64])
{'n': 598, 'top1': 0.54347825050354, 'pwr_loss_db_top1': 0.11562828719615936, 'overhead_top1': 0.015625, 'top3': 0.8494983315467834, 'pwr_loss_db_top3': 0.062350038439035416, 'overhead_top3': 0.046875, 'top5': 0.928093671798706, 'pwr_loss_db_top5': 0.044501893222332, 'overhead_top5': 0.078125, 'K90': 5, 'K95': 6, 'economie_K90': 0.921875}


chargement (avec l’ordre caméra), modèle, entraînement

In [13]:
# ---- ordre des fichiers caméra : lignes croissantes de master.csv à l'intérieur de chaque split
def master_order(split):
    return sorted((int(i) for i in splits[split]), key=lambda i: pos[i])

def cam_emb(seed, split):                                       # float16 -> float32 avant toute statistique
    return np.load(CAM_DIR / f"seed{seed}_{split}_embeddings.npy").astype(np.float32)

def cam_logits(seed, split):                                    # réordonné selon splits[split]
    L = np.load(CAM_DIR / f"seed{seed}_{split}_logits.npy")
    p = {i: k for k, i in enumerate(master_order(split))}
    return L[[p[int(i)] for i in splits[split]]]

def gps_logits(seed, split):
    return np.load(GPS_RES_DIR / f"seed{seed}_{split}_logits.npy")

def load_gps(seed):        # 128 dimensions
    return {s: (np.load(GPS_EMB_DIR / f"{s}_idx.npy"), np.load(GPS_EMB_DIR / f"seed{seed}_{s}_emb.npy"))
            for s in ["train", "val", "test"]}

def load_camera(seed):     # 1280 dimensions
    return {s: (np.array(master_order(s)), cam_emb(seed, s)) for s in ["train", "val", "test"]}

def align(mods, split):
    ids = np.array(splits[split])
    out = []
    for m in mods:
        idx, emb = m[split]
        p = {int(i): k for k, i in enumerate(idx)}
        out.append(emb[[p[int(i)] for i in ids]])
    return out

def get_data(seed):
    gps, cam = load_gps(seed), load_camera(seed)
    D = {s: align([gps, cam], s) for s in ["train", "val", "test"]}
    stats = [(x.mean(0), x.std(0) + 1e-6) for x in D["train"]]            # statistiques du train seulement
    return {s: [torch.tensor((x - m) / sd, dtype=torch.float32) for x, (m, sd) in zip(D[s], stats)]
            for s in D}

class AttnFusion(nn.Module):
    """Un jeton par modalité ; l'attention fait regarder chaque modalité aux autres."""
    def __init__(self, dims, d=64, heads=4, layers=1, dropout=0.2, n_classes=N_BEAMS):
        super().__init__()
        self.proj = nn.ModuleList([nn.Sequential(nn.Linear(di, d), nn.LayerNorm(d)) for di in dims])
        self.mod_emb = nn.Parameter(torch.randn(len(dims), d) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d, nhead=heads, dim_feedforward=2 * d,
                                           dropout=dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d * len(dims)), nn.Dropout(dropout),
                                  nn.Linear(d * len(dims), n_classes))
    def forward(self, xs):
        toks = torch.stack([p(x) for p, x in zip(self.proj, xs)], dim=1) + self.mod_emb
        return self.head(self.enc(toks).flatten(1))

def soft_targets(Pb, Yb, tau):
    q = Pb / Pb.max(1, keepdim=True).values.clamp(min=1e-9)
    t = F.softmax(q / tau, dim=1)
    ok = Pb.max(1).values > 0
    return torch.where(ok[:, None], t, F.one_hot(Yb, N_BEAMS).float())

def predict(model, xs, bs=512):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(xs[0]), bs):
            out.append(model([x[i:i + bs].to(dev) for x in xs]).cpu())
    return torch.cat(out)

def train_one(cfg, data, seed, max_epochs=60, patience=8, bs=128, lr=1e-3):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    dims = [x.shape[1] for x in data["train"]]
    model = AttnFusion(dims, d=cfg["d"], heads=cfg["heads"], layers=cfg["layers"],
                       dropout=cfg["dropout"]).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    Xtr, Ytr, Ptr = data["train"], Y["train"], P["train"]
    n, best, best_state, bad, t0 = len(Ytr), None, None, 0, time.time()
    for ep in range(max_epochs):
        model.train(); perm = torch.randperm(n)
        for i in range(0, n, bs):
            b = perm[i:i + bs]
            logits = model([x[b].to(dev) for x in Xtr])
            if cfg["tau"] is None:
                loss = F.cross_entropy(logits, Ytr[b].to(dev))
            else:
                tgt = soft_targets(Ptr[b], Ytr[b], cfg["tau"]).to(dev)
                loss = -(tgt * F.log_softmax(logits, 1)).sum(1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        mv = metrics_from_logits(predict(model, data["val"]), Y["val"], P["val"])
        score = (mv["top3"], -mv["pwr_loss_db_top1"])            # règle : Top-3 de validation, puis perte de puissance
        if best is None or score > best:
            best, best_state, bad = score, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, time.time() - t0

Cellule 3 : contrôles (à lancer avant tout entraînement)

In [14]:
# 1) splits.json est-il déjà dans l'ordre master.csv ? (information, pas bloquant)
for s in ["train", "val", "test"]:
    r = [pos[int(i)] for i in splits[s]]
    print(s, "splits déjà en ordre croissant de master :", bool(np.all(np.diff(r) > 0)))

# 2) les embeddings par split correspondent-ils aux lignes de l'embedding global ? (valide l'ordre supposé)
full = np.load(CAM_DIR / "seed0_embeddings.npy").astype(np.float32)
for s in ["train", "val", "test"]:
    rows = [pos[i] for i in master_order(s)]
    print(s, "emb split == emb global :", np.allclose(full[rows], cam_emb(0, s), atol=1e-3))

# 3) formes
d0 = get_data(0)
print("train :", [tuple(x.shape) for x in d0["train"]])   # attendu : [(2781, 128), (2781, 1280)]
print("test  :", [tuple(x.shape) for x in d0["test"]])    # attendu : [(598, 128), (598, 1280)]

# 4) la caméra seule retrouve les chiffres de seed0.json avec NOTRE ordre ?
mc = full_metrics(torch.tensor(cam_logits(0, "test")), Y["test"], P["test"])
print("Caméra seule (test, seed 0) :", {k: round(mc[k], 4) for k in ("top1", "top3", "top5")})
# Attendu : top1=0.4415, top3=0.7408, top5=0.8177

# 5) hypothèse « embeddings trop favorables sur le train » : caméra train contre test (seed 0)
mtr = full_metrics(torch.tensor(cam_logits(0, "train")), Y["train"], P["train"])
print("Caméra train : top1 =", round(mtr["top1"], 3), "top3 =", round(mtr["top3"], 3),
      "| test : top1 =", round(mc["top1"], 3), "top3 =", round(mc["top3"], 3))

train splits déjà en ordre croissant de master : True
val splits déjà en ordre croissant de master : True
test splits déjà en ordre croissant de master : True
train emb split == emb global : True
val emb split == emb global : True
test emb split == emb global : True
train : [(2781, 128), (2781, 1280)]
test  : [(598, 128), (598, 1280)]
Caméra seule (test, seed 0) : {'top1': 0.4415, 'top3': 0.7408, 'top5': 0.8177}
Caméra train : top1 = 0.555 top3 = 0.891 | test : top1 = 0.441 top3 = 0.741


recherche de configuration (validation, seed 0)

In [15]:
configs = [
    dict(d=64,  heads=4, layers=1, dropout=0.2, tau=None),
    dict(d=64,  heads=4, layers=2, dropout=0.2, tau=None),
    dict(d=128, heads=4, layers=1, dropout=0.3, tau=None),
    dict(d=64,  heads=4, layers=1, dropout=0.2, tau=0.05),
    dict(d=128, heads=4, layers=2, dropout=0.3, tau=0.05),
    dict(d=64,  heads=4, layers=1, dropout=0.5, tau=0.05),
]
data0 = get_data(0)
rows = []
for k, cfg in enumerate(configs):
    model, tt = train_one(cfg, data0, seed=0)
    mv = full_metrics(predict(model, data0["val"]), Y["val"], P["val"])
    rows.append({"cfg": k, **cfg, "params": count_params(model), "train_time_s": round(tt, 1),
                 "val_top1": mv["top1"], "val_top3": mv["top3"], "val_top5": mv["top5"],
                 "val_pwr_loss_db_top1": mv["pwr_loss_db_top1"], "val_K90": mv["K90"]})
    print(rows[-1])

val_table = pd.DataFrame(rows).sort_values(["val_top3", "val_pwr_loss_db_top1"], ascending=[False, True])
val_table.to_csv(OUT / "val_configs.csv", index=False)
print(val_table.round(4).to_string())
best_cfg = configs[int(val_table.iloc[0]["cfg"])]
print("Meilleure configuration (validation) :", best_cfg)

{'cfg': 0, 'd': 64, 'heads': 4, 'layers': 1, 'dropout': 0.2, 'tau': None, 'params': 132608, 'train_time_s': 4.8, 'val_top1': 0.31893688440322876, 'val_top3': 0.6727574467658997, 'val_top5': 0.7857142686843872, 'val_pwr_loss_db_top1': 0.3181609511375427, 'val_K90': 15}
{'cfg': 1, 'd': 64, 'heads': 4, 'layers': 2, 'dropout': 0.2, 'tau': None, 'params': 166080, 'train_time_s': 4.8, 'val_top1': 0.3255814015865326, 'val_top3': 0.6561461687088013, 'val_top5': 0.7740863561630249, 'val_pwr_loss_db_top1': 0.30323004722595215, 'val_K90': 14}
{'cfg': 2, 'd': 128, 'heads': 4, 'layers': 1, 'dropout': 0.3, 'tau': None, 'params': 330688, 'train_time_s': 4.7, 'val_top1': 0.3222591280937195, 'val_top3': 0.6578072905540466, 'val_top5': 0.7956810593605042, 'val_pwr_loss_db_top1': 0.27801597118377686, 'val_K90': 15}
{'cfg': 3, 'd': 64, 'heads': 4, 'layers': 1, 'dropout': 0.2, 'tau': 0.05, 'params': 132608, 'train_time_s': 3.6, 'val_top1': 0.2923588156700134, 'val_top3': 0.6810631155967712, 'val_top5': 0.8

In [18]:
res = []
for seed in [0, 1, 2]:
    data = get_data(seed)
    model, tt = train_one(best_cfg, data, seed=seed)
    Lv, Lt = predict(model, data["val"]), predict(model, data["test"])
    mv = full_metrics(Lv, Y["val"], P["val"])
    mt = full_metrics(Lt, Y["test"], P["test"])               # test évalué une seule fois
    rec = {"exp_id": "FUS-F4c", "seed": seed,
           **{f"val_{k}": v for k, v in mv.items()},
           **{f"test_{k}": v for k, v in mt.items()},
           "params": count_params(model), "train_time_s": round(tt, 1), "config": best_cfg}
    json.dump(rec, open(OUT / f"seed{seed}.json", "w"), indent=1)
    np.save(OUT / f"seed{seed}_val_logits.npy", Lv.numpy())
    np.save(OUT / f"seed{seed}_test_logits.npy", Lt.numpy())
    torch.save(model.state_dict(), OUT / f"seed{seed}_best.pt")
    res.append(rec)
    print(seed, "| val top3 =", round(mv["top3"], 4), "| test top1/top3/top5 =",
          round(mt["top1"], 4), round(mt["top3"], 4), round(mt["top5"], 4), "| K90 test =", mt["K90"])

pd.DataFrame(res).drop(columns="config").to_csv(OUT / "final_seeds.csv", index=False)

def ref(loader, split):
    return pd.DataFrame([full_metrics(torch.tensor(loader(s, split)), Y[split], P[split]) for s in [0, 1, 2]])

cols = ["top1", "top3", "top5", "pwr_loss_db_top1", "pwr_loss_db_top3", "pwr_loss_db_top5", "K90", "K95", "economie_K90"]
for split in ["val", "test"]:
    fus = pd.DataFrame([{c: r[f"{split}_{c}"] for c in cols} for r in res])
    refs = {"GPS-E3 seul": ref(gps_logits, split)[cols],
            "CAM-E3 seule": ref(cam_logits, split)[cols],
            "FUS-F4c (GPS+caméra)": fus}
    tab = pd.DataFrame({k: {c: f"{v[c].mean():.3f} ± {v[c].std():.3f}" for c in cols}
                        for k, v in refs.items()}).T
    print(f"\n=== {split.upper()} (moyenne ± écart-type sur 3 seeds) ===")
    print(tab.to_string())
    tab.to_csv(OUT / f"tableau_comparaison_{split}.csv")

0 | val top3 = 0.686 | test top1/top3/top5 = 0.4181 0.8077 0.9097 | K90 test = 5
1 | val top3 = 0.6877 | test top1/top3/top5 = 0.505 0.8127 0.9181 | K90 test = 5
2 | val top3 = 0.6877 | test top1/top3/top5 = 0.3378 0.7742 0.9231 | K90 test = 5

=== VAL (moyenne ± écart-type sur 3 seeds) ===
                               top1           top3           top5 pwr_loss_db_top1 pwr_loss_db_top3 pwr_loss_db_top5             K90             K95   economie_K90
GPS-E3 seul           0.381 ± 0.003  0.732 ± 0.009  0.843 ± 0.005    0.166 ± 0.003    0.087 ± 0.000    0.069 ± 0.000  11.000 ± 0.000  21.667 ± 0.577  0.828 ± 0.000
CAM-E3 seule          0.287 ± 0.009  0.560 ± 0.004  0.678 ± 0.016    0.528 ± 0.052    0.324 ± 0.047    0.216 ± 0.037  25.000 ± 4.000  40.333 ± 5.033  0.609 ± 0.062
FUS-F4c (GPS+caméra)  0.343 ± 0.023  0.687 ± 0.001  0.812 ± 0.009    0.215 ± 0.027    0.103 ± 0.003    0.075 ± 0.005  13.667 ± 0.577  29.667 ± 4.041  0.786 ± 0.009

=== TEST (moyenne ± écart-type sur 3 seeds) ===
   

# FUS-F4c — Fusion GPS + caméra par attention croisée

## Hypothèse
La caméra voit la scène et le GPS donne la position (x, y) du véhicule. **Hypothèse** : les fusionner par attention croisée dépasse le GPS seul en exploitant une information que la position ne contient pas.

## Ce qui a été fait
- Entrées : embeddings GPS-E3 (128 dim.) et CAM-E3 (1280 dim., float16 converti en float32), normalisés avec les statistiques du train.
- Modèle : un jeton par modalité, encodeur Transformer, sortie sur 64 beams. Mêmes splits, même protocole et même grille de 6 configurations que FUS-F4a.
- Configuration retenue (validation, seed 0) : d = 64, 1 couche, 4 têtes, dropout 0,5, labels doux tau = 0,05 ; 132 608 paramètres.
- Les 6 configurations ont un Top-3 de validation compris entre 0,656 et 0,686 : l'écart n'est pas significatif.
- Évaluation : seeds 0, 1, 2 ; test évalué une seule fois.

## Résultats (test, moyenne ± écart-type sur 3 seeds)

| | Top-1 | Top-3 | Top-5 | Perte puiss. Top-1 (dB) | K90 | K95 | Économie K90 |
|---|---|---|---|---|---|---|---|
| GPS-E3 seul | 0,556 ± 0,013 | 0,852 ± 0,003 | 0,928 ± 0,004 | 0,112 ± 0,004 | 5 | 6,7 ± 0,6 | 92,2 % |
| CAM-E3 seule | 0,384 ± 0,114 | 0,684 ± 0,094 | 0,805 ± 0,017 | 0,508 ± 0,082 | 10,3 ± 1,2 | 17,3 ± 0,6 | 83,9 % |
| **FUS-F4c** | 0,420 ± 0,084 | 0,798 ± 0,021 | 0,917 ± 0,007 | 0,195 ± 0,057 | 5 | 7,7 ± 1,2 | 92,2 % |

Validation : Top-3 de 0,732 (GPS), 0,560 (caméra) et 0,687 (fusion), avec un K90 de 11, 25 et 13,7. Le classement est le même qu'en test, mais la fusion n'égale pas le GPS à K90 en validation.

## Conclusion
- **La fusion est nettement meilleure que la caméra seule** (+11 points en Top-3 et Top-5, K90 de 10,3 à 5).
- **Elle reste inférieure au GPS seul** : -13,6 points en Top-1, -5,4 en Top-3, -1,1 en Top-5, soit un écart qui diminue quand K augmente. À K90, les deux testent 5 beams sur 64 (économie de 92,2 %).
- **Hypothèse non confirmée** : nous n'observons pas de gain par rapport au GPS seul.
- Le Top-1 est instable d'une seed à l'autre (0,418 ; 0,505 ; 0,338) : rapporter la moyenne ± écart-type, et privilégier le Top-3 et le Top-5.


# FUS-F4c — GPS + caméra (attention croisée)
- Hypothèse : fusionner GPS et caméra dépasse le GPS seul.
- Changement : embeddings GPS-E3 (128) + CAM-E3 (1280) ; d=64, 1 couche, dropout 0,5, labels doux ; 3 seeds.
- Résultat : test Top-3 0,798 (GPS seul 0,852 ; caméra seule 0,684) ; K90 = 5 pour fusion et GPS.
- Échec : la fusion reste sous le GPS seul ; la complémentarité n'est pas démontrée (cf. cellule 6).